# Analise de campos e da coluna `value` - RetailRocket

Este notebook complementa a EDA inicial com um estudo da estrutura da coluna `value` de `item_properties`. A intencao aqui nao e repetir volume, funil ou sparsity do notebook 01, e sim transformar a documentacao do RetailRocket em uma gramatica pratica para interpretar valores hasheados, numericos, multivalorados e propriedades especiais.

Referencia do dataset: https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset

Objetivos:

- Registrar rapidamente o papel de cada tabela e campo.
- Explicar a gramatica documentada do `value`: excecoes, hashes, numeros codificados e textos tokenizados.
- Validar essa gramatica nos dados reais com exemplos e contagens por tipo de valor.

## 0. Preparacao

Por padrao, os CSVs sao lidos de `data/raw/retailrocket/`. Para usar outro caminho, defina a variavel de ambiente `RETAILROCKET_DATA_DIR` antes de abrir o notebook.

Por padrao, o notebook le uma amostra grande de cada arquivo de propriedades para manter a exploracao interativa. Para ler tudo, altere `PROPERTY_NROWS` para `None` ou aumente `RETAILROCKET_PROPERTY_NROWS`. As analises pesadas tambem usam amostras: `RETAILROCKET_FIELD_PROFILE_SAMPLE_ROWS`, `RETAILROCKET_VALUE_ANALYSIS_ROWS`, `RETAILROCKET_TOKEN_SAMPLE_ROWS` e `RETAILROCKET_STABILITY_SAMPLE_ROWS`.

In [56]:
from __future__ import annotations

import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def env_int(name: str, default: int | None = None) -> int | None:
    value = os.environ.get(name)
    if value in (None, ''):
        return default
    return int(value)


def sample_frame(
    frame: pd.DataFrame,
    max_rows: int | None,
    random_state: int = 42,
) -> pd.DataFrame:
    if max_rows is None or len(frame) <= max_rows:
        return frame.copy()
    return frame.sample(max_rows, random_state=random_state).copy()


NUMERIC_TOKEN_PATTERN = r'\bn-?\d+\.\d{3}\b'
PLAIN_HASH_TOKEN_PATTERN = r'(?<![\w.])\d+(?![\w.])'


def examples_text(values: pd.Series, limit: int = 6) -> str:
    examples = values.dropna().astype(str).drop_duplicates().head(limit).tolist()
    return ' | '.join(examples)


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = Path(
    os.environ.get('RETAILROCKET_DATA_DIR', PROJECT_ROOT / 'data' / 'raw' / 'retailrocket')
).expanduser()

events_path = DATA_DIR / 'events.csv'
category_tree_path = DATA_DIR / 'category_tree.csv'
item_property_paths = [
    DATA_DIR / 'item_properties_part1.csv',
    DATA_DIR / 'item_properties_part2.csv',
]
csv_paths = [events_path, category_tree_path, *item_property_paths]

missing_files = [path.name for path in csv_paths if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        f"CSV(s) nao encontrados: {', '.join(missing_files)}. "
        'Ajuste RETAILROCKET_DATA_DIR ou coloque os arquivos em data/raw/retailrocket/.'
    )

PROPERTY_NROWS = env_int('RETAILROCKET_PROPERTY_NROWS', 500_000)
FIELD_PROFILE_SAMPLE_ROWS = env_int('RETAILROCKET_FIELD_PROFILE_SAMPLE_ROWS', 250_000)
VALUE_ANALYSIS_ROWS = env_int('RETAILROCKET_VALUE_ANALYSIS_ROWS', 500_000)
TOKEN_SAMPLE_ROWS = env_int('RETAILROCKET_TOKEN_SAMPLE_ROWS', 200_000)
STABILITY_SAMPLE_ROWS = env_int('RETAILROCKET_STABILITY_SAMPLE_ROWS', 500_000)

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Inventario dos arquivos

A primeira checagem confirma quais tabelas existem, seu tamanho e quais colunas aparecem no cabecalho. Isso ajuda a manter o notebook alinhado com a documentacao do Kaggle e detecta rapidamente arquivos incompletos ou fora do padrao.

In [57]:
table_paths = {
    'events': events_path,
    'category_tree': category_tree_path,
    'item_properties_part1': item_property_paths[0],
    'item_properties_part2': item_property_paths[1],
}

inventory_rows = []
for table_name, path in table_paths.items():
    sample = pd.read_csv(path, nrows=5)
    inventory_rows.append(
        {
            'tabela': table_name,
            'arquivo': path.name,
            'tamanho_mib': round(path.stat().st_size / 1024**2, 2),
            'colunas': ', '.join(sample.columns),
        }
    )

display(pd.DataFrame(inventory_rows))

,tabela,arquivo,tamanho_mib,colunas
0,events,events.csv,89.87,"timestamp, visitorid, event, itemid, transactionid"
1,category_tree,category_tree.csv,0.01,"categoryid, parentid"
2,item_properties_part1,item_properties_part1.csv,461.88,"timestamp, itemid, property, value"
3,item_properties_part2,item_properties_part2.csv,389.99,"timestamp, itemid, property, value"


## 2. Leitura das tabelas

`item_properties_part1` e `item_properties_part2` sao unidos somente em memoria. A coluna `timestamp` de eventos vira `event_time`, e a de propriedades vira `property_time`, para evitar ambiguidade nas proximas analises.

In [58]:
events = pd.read_csv(events_path)
events['event_time'] = pd.to_datetime(events['timestamp'], unit='ms', utc=True)

category_tree = pd.read_csv(category_tree_path)

item_properties = pd.concat(
    [pd.read_csv(path, nrows=PROPERTY_NROWS) for path in item_property_paths],
    ignore_index=True,
)
item_properties['property_time'] = pd.to_datetime(
    item_properties['timestamp'], unit='ms', utc=True
)
item_properties['property'] = item_properties['property'].astype('string')
item_properties['value'] = item_properties['value'].astype('string')

shape_summary = pd.DataFrame(
    {
        'tabela': ['events', 'category_tree', 'item_properties'],
        'linhas': [len(events), len(category_tree), len(item_properties)],
        'colunas': [events.shape[1], category_tree.shape[1], item_properties.shape[1]],
    }
)

display(shape_summary)
display(events.head())
display(category_tree.head())
display(item_properties.head())

,tabela,linhas,colunas
0,events,2756101,6
1,category_tree,1669,2
2,item_properties,1000000,5


,timestamp,visitorid,event,itemid,transactionid,event_time
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117000+00:00
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164000+00:00
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827000+00:00
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914000+00:00
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106000+00:00


,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


,timestamp,itemid,property,value,property_time
0,1435460400000,460429,categoryid,1338,2015-06-28 03:00:00+00:00
1,1441508400000,206783,888,1116713 960601 n277.200,2015-09-06 03:00:00+00:00
2,1439089200000,395014,400,n552.000 639502 n720.000 424566,2015-08-09 03:00:00+00:00
3,1431226800000,59481,790,n15360.000,2015-05-10 03:00:00+00:00
4,1431831600000,156781,917,828513,2015-05-17 03:00:00+00:00


## 3. Dicionario e perfil de campos

A tabela abaixo combina uma leitura semantica do dataset com estatisticas observadas localmente. A coluna `uso_recomendacao` antecipa como cada campo pode entrar em uma base de treino.

In [59]:
field_catalog = pd.DataFrame(
    [
        ('events', 'timestamp', 'momento do evento em milissegundos', 'split cronologico e recencia', 'alto se usado fora de ordem'),
        ('events', 'visitorid', 'identificador anonimo do usuario', 'chave de historico e embedding', 'baixo'),
        ('events', 'event', 'tipo de acao: view, addtocart ou transaction', 'label, peso implicito e funil', 'medio se label e feature misturarem'),
        ('events', 'itemid', 'identificador anonimo do item', 'chave de catalogo e embedding', 'baixo'),
        ('events', 'transactionid', 'identificador de compra quando existe', 'label de conversao', 'alto se usado como feature'),
        ('item_properties', 'timestamp', 'momento em que a propriedade vale', 'join temporal point-in-time', 'alto se usar valor futuro'),
        ('item_properties', 'itemid', 'identificador anonimo do item', 'chave para unir com eventos', 'baixo'),
        ('item_properties', 'property', 'nome ou hash da propriedade', 'selecionar grupos de atributos', 'medio'),
        ('item_properties', 'value', 'valor da propriedade, muitas vezes hasheado', 'feature de conteudo ou categoria', 'medio/alto'),
        ('category_tree', 'categoryid', 'categoria do item', 'feature hierarquica de catalogo', 'baixo/medio'),
        ('category_tree', 'parentid', 'categoria pai', 'agregacao por familia de itens', 'baixo'),
    ],
    columns=['tabela', 'campo', 'leitura_documentacao', 'uso_recomendacao', 'risco_vazamento'],
)


def profile_fields(
    table_name: str,
    frame: pd.DataFrame,
    sample_rows: int | None = None,
) -> pd.DataFrame:
    profile_frame = sample_frame(frame, sample_rows)
    rows = []
    for column in profile_frame.columns:
        series = profile_frame[column]
        examples = series.dropna().astype(str).drop_duplicates().head(5).tolist()
        rows.append(
            {
                'tabela': table_name,
                'campo': column,
                'dtype': str(series.dtype),
                'linhas_tabela': len(frame),
                'linhas_perfil': len(profile_frame),
                'perfil_amostral': len(profile_frame) < len(frame),
                'nulos': int(series.isna().sum()),
                'cardinalidade': int(series.nunique(dropna=True)),
                'exemplos': ' | '.join(examples),
            }
        )
    return pd.DataFrame(rows)


field_profiles = pd.concat(
    [
        profile_fields('events', events),
        profile_fields('category_tree', category_tree),
        profile_fields('item_properties', item_properties, FIELD_PROFILE_SAMPLE_ROWS),
    ],
    ignore_index=True,
)

field_analysis = field_profiles.merge(field_catalog, on=['tabela', 'campo'], how='left')
display(
    Markdown(
        '`item_properties` usa perfil amostral quando a tabela carregada passa de '
        f'{FIELD_PROFILE_SAMPLE_ROWS:,} linhas. Ajuste `RETAILROCKET_FIELD_PROFILE_SAMPLE_ROWS` '
        'para aumentar ou reduzir essa amostra.'
    )
)
display(field_analysis)

`item_properties` usa perfil amostral quando a tabela carregada passa de 250,000 linhas. Ajuste `RETAILROCKET_FIELD_PROFILE_SAMPLE_ROWS` para aumentar ou reduzir essa amostra.

,tabela,campo,dtype,linhas_tabela,linhas_perfil,perfil_amostral,nulos,cardinalidade,exemplos,leitura_documentacao,uso_recomendacao,risco_vazamento
0,events,timestamp,int64,2756101,2756101,False,0,2750455,1433221332117 | 1433224214164 | 1433221999827 | 1433221955914 | 1433221337106,momento do evento em milissegundos,split cronologico e recencia,alto se usado fora de ordem
1,events,visitorid,int64,2756101,2756101,False,0,1407580,257597 | 992329 | 111016 | 483717 | 951259,identificador anonimo do usuario,chave de historico e embedding,baixo
2,events,event,str,2756101,2756101,False,0,3,view | addtocart | transaction,"tipo de acao: view, addtocart ou transaction","label, peso implicito e funil",medio se label e feature misturarem
3,events,itemid,int64,2756101,2756101,False,0,235061,355908 | 248676 | 318965 | 253185 | 367447,identificador anonimo do item,chave de catalogo e embedding,baixo
4,events,transactionid,float64,2756101,2756101,False,2733644,17672,4000.0 | 11117.0 | 5444.0 | 13556.0 | 7244.0,identificador de compra quando existe,label de conversao,alto se usado como feature
5,events,event_time,"datetime64[ms, UTC]",2756101,2756101,False,0,2750455,2015-06-02 05:02:12.117000+00:00 | 2015-06-02 05:50:14.164000+00:00 | 2015-06-02 05:13:19.827000+00:00 | 2015-06-02 ...,NaN,NaN,NaN
6,category_tree,categoryid,int64,1669,1669,False,0,1669,1016 | 809 | 570 | 1691 | 536,categoria do item,feature hierarquica de catalogo,baixo/medio
7,category_tree,parentid,float64,1669,1669,False,25,362,213.0 | 169.0 | 9.0 | 885.0 | 1691.0,categoria pai,agregacao por familia de itens,baixo
8,item_properties,timestamp,int64,1000000,250000,True,0,18,1442113200000 | 1438484400000 | 1439089200000 | 1431226800000 | 1440903600000,momento em que a propriedade vale,join temporal point-in-time,alto se usar valor futuro
9,item_properties,itemid,int64,1000000,250000,True,0,173164,156843 | 363803 | 82250 | 318800 | 426628,identificador anonimo do item,chave para unir com eventos,baixo


## 4. Gramatica documentada do `value`

A documentacao do Kaggle deixa claro que `value` nao é uma coluna homogenea. Ela mistura excecoes legiveis, tokens de texto hasheados e numeros codificados. Esta é a regra de leitura que o restante do notebook valida nos dados.

In [60]:
value_grammar_docs = pd.DataFrame(
    [
        (
            'categoryid',
            'excecao nao hasheada',
            'valor inteiro com o identificador da categoria do item',
            '`1338`',
            'usar como feature de catalogo e ligar com `category_tree`',
        ),
        (
            'available',
            'excecao nao hasheada',
            'flag de disponibilidade do item',
            '`1` disponivel, `0` indisponivel',
            'usar somente com timestamp anterior ao evento',
        ),
        (
            'propriedades numericas',
            'numero codificado',
            'todo numero recebe prefixo `n` e 3 casas decimais',
            '`n5.000`, `n-3.675`, `n277.200`',
            'extrair como valor numerico depois de remover `n`',
        ),
        (
            'propriedades textuais',
            'texto normalizado e hasheado',
            'palavras passam por stemming e viram ids inteiros anonimos',
            '`24214 44214 n2017.000`',
            'tratar como bag-of-tokens ou atributos categoricos multivalorados',
        ),
        (
            'qualquer propriedade',
            'changelog temporal',
            'linhas aparecem quando um valor muda; constantes podem aparecer uma vez',
            'mesmo item/property em datas diferentes',
            'fazer join point-in-time, nunca usar ultimo valor global',
        ),
    ],
    columns=['escopo', 'tipo', 'regra', 'exemplo', 'uso_para_modelagem'],
)

property_counts = item_properties['property'].value_counts().head(20).rename('linhas').to_frame()

display(value_grammar_docs)
display(Markdown('Top propriedades so para orientar a investigacao do `value`.'))
display(property_counts)

,escopo,tipo,regra,exemplo,uso_para_modelagem
0,categoryid,excecao nao hasheada,valor inteiro com o identificador da categoria do item,`1338`,usar como feature de catalogo e ligar com `category_tree`
1,available,excecao nao hasheada,flag de disponibilidade do item,"`1` disponivel, `0` indisponivel",usar somente com timestamp anterior ao evento
2,propriedades numericas,numero codificado,todo numero recebe prefixo `n` e 3 casas decimais,"`n5.000`, `n-3.675`, `n277.200`",extrair como valor numerico depois de remover `n`
3,propriedades textuais,texto normalizado e hasheado,palavras passam por stemming e viram ids inteiros anonimos,`24214 44214 n2017.000`,tratar como bag-of-tokens ou atributos categoricos multivalorados
4,qualquer propriedade,changelog temporal,linhas aparecem quando um valor muda; constantes podem aparecer uma vez,mesmo item/property em datas diferentes,"fazer join point-in-time, nunca usar ultimo valor global"


Top propriedades so para orientar a investigacao do `value`.

,linhas
property,
888,148335
790,88844
available,73869
categoryid,38905
6,31512
283,29530
776,28196
678,23561
364,23219


## 5. Anatomia real do `value`

A partir da gramatica documentada, cada linha recebe uma classificacao de formato. Essa classificacao é mais util do que olhar apenas cardinalidade, porque diz como o valor deve ser transformado em feature.

In [61]:
value_frame = sample_frame(
    item_properties[['itemid', 'property', 'value', 'property_time']],
    VALUE_ANALYSIS_ROWS,
)
value_frame['value_text'] = value_frame['value'].fillna('').astype('string')
value_frame['token_count'] = value_frame['value_text'].str.split().str.len().fillna(0).astype(int)
value_frame['encoded_numeric_tokens'] = value_frame['value_text'].str.count(NUMERIC_TOKEN_PATTERN)
value_frame['plain_hash_tokens'] = value_frame['value_text'].str.count(PLAIN_HASH_TOKEN_PATTERN)
value_frame['other_tokens'] = (
    value_frame['token_count']
    - value_frame['encoded_numeric_tokens']
    - value_frame['plain_hash_tokens']
)
value_frame['is_categoryid'] = value_frame['property'].eq('categoryid')
value_frame['is_available'] = value_frame['property'].eq('available')
value_frame['is_multitoken'] = value_frame['token_count'].gt(1)

value_frame['value_grammar'] = 'other_or_unexpected'
value_frame.loc[value_frame['is_categoryid'], 'value_grammar'] = 'special_categoryid'
value_frame.loc[value_frame['is_available'], 'value_grammar'] = 'special_available'
mask_regular = ~(value_frame['is_categoryid'] | value_frame['is_available'])
mask_numeric_only = mask_regular & value_frame['encoded_numeric_tokens'].eq(value_frame['token_count'])
mask_plain_only = mask_regular & value_frame['plain_hash_tokens'].eq(value_frame['token_count'])
mask_mixed = mask_regular & value_frame['encoded_numeric_tokens'].gt(0) & value_frame['plain_hash_tokens'].gt(0)
value_frame.loc[mask_numeric_only, 'value_grammar'] = 'encoded_numeric_only'
value_frame.loc[mask_plain_only & value_frame['token_count'].eq(1), 'value_grammar'] = 'single_hashed_text_token'
value_frame.loc[mask_plain_only & value_frame['token_count'].gt(1), 'value_grammar'] = 'multi_hashed_text_tokens'
value_frame.loc[mask_mixed, 'value_grammar'] = 'mixed_hashed_text_and_numeric'

grammar_summary = (
    value_frame.groupby('value_grammar')
    .agg(
        linhas=('value_text', 'size'),
        propriedades=('property', 'nunique'),
        itens=('itemid', 'nunique'),
        tokens_media=('token_count', 'mean'),
        exemplos=('value_text', examples_text),
    )
    .sort_values('linhas', ascending=False)
)
grammar_summary['percentual'] = (grammar_summary['linhas'] / len(value_frame) * 100).round(2)
grammar_summary['tokens_media'] = grammar_summary['tokens_media'].round(3)

display(
    Markdown(
        f'Classificacao usando **{len(value_frame):,} linhas** de '
        f'**{len(item_properties):,} linhas** carregadas de `item_properties`. '
        'Ajuste `RETAILROCKET_VALUE_ANALYSIS_ROWS` para mudar a amostra.'
    )
)
display(grammar_summary)

Classificacao usando **500,000 linhas** de **1,000,000 linhas** carregadas de `item_properties`. Ajuste `RETAILROCKET_VALUE_ANALYSIS_ROWS` para mudar a amostra.

,linhas,propriedades,itens,tokens_media,exemplos,percentual
value_grammar,,,,,,
single_hashed_text_token,205811,797,153352,1.000,409699 | 769062 | 1037891 | 341599 | 1301149 | 591638,41.16
multi_hashed_text_tokens,109887,378,81060,4.065,298888 1318567 | 150169 431733 1037547 | 1141052 140719 553394 | 741149 1186729 474154 107828 1118082 | 911286 12845...,21.98
mixed_hashed_text_and_numeric,70483,352,49970,7.080,428970 65916 6399 n111816.000 373898 784581 1297729 n36.000 350726 30603 832471 | n144.000 396934 | 1067666 n16284.0...,14.10
encoded_numeric_only,57205,153,43498,1.004,n11280.000 | n88320.000 | n12120.000 | n58980.000 | n207.600 | n204000.000,11.44
special_available,37137,1,27749,1.000,0 | 1,7.43
special_categoryid,19473,1,16269,1.000,1084 | 498 | 546 | 1674 | 1561 | 421,3.89
other_or_unexpected,4,3,4,2.000,nInfinity | 42654 nInfinity | 12092 nInfinity | 1217057 633814 nInfinity,0.00


## 6. Tokens: texto hasheado vs numero codificado

Depois de classificar linhas, vale explodir tokens. Pela documentacao, tokens `n...` representam numeros reais codificados; inteiros comuns representam palavras normalizadas e hasheadas, exceto nas propriedades especiais.

In [62]:
def classify_token(token: str) -> str:
    normalized = token.strip().lower()
    if re.fullmatch(r'n-?\d+\.\d{3}', normalized):
        return 'encoded_numeric_token'
    if re.fullmatch(r'\d+', normalized):
        return 'hashed_text_token'
    if re.fullmatch(r'n-?\d+(\.\d+)?', normalized):
        return 'numeric_token_variant'
    return 'unexpected_token'


token_sample_size = min(TOKEN_SAMPLE_ROWS or len(value_frame), len(value_frame))
token_sample = value_frame.sample(token_sample_size, random_state=42)
tokens = (
    token_sample[['property', 'value_grammar', 'value_text']]
    .assign(token=lambda frame: frame['value_text'].str.split())
    .explode('token')
    .dropna(subset=['token'])
)
tokens['token_type'] = tokens['token'].map(classify_token)

token_type_summary = (
    tokens.groupby('token_type')
    .agg(
        tokens=('token', 'size'),
        tokens_unicos=('token', 'nunique'),
        propriedades=('property', 'nunique'),
        exemplos=('token', examples_text),
    )
    .sort_values('tokens', ascending=False)
)
token_type_summary['percentual'] = (
    token_type_summary['tokens'] / token_type_summary['tokens'].sum() * 100
).round(2)

top_tokens_by_type = (
    tokens.groupby(['token_type', 'token'])
    .size()
    .rename('frequencia')
    .reset_index()
    .sort_values(['token_type', 'frequencia'], ascending=[True, False])
    .groupby('token_type')
    .head(10)
)

display(Markdown(f'Amostra usada para tokens: **{token_sample_size:,} linhas**.'))
display(token_type_summary)
display(top_tokens_by_type)

Amostra usada para tokens: **200,000 linhas**.

,tokens,tokens_unicos,propriedades,exemplos,percentual
token_type,,,,,
hashed_text_token,436918,56675,876,1285872 | 397563 | 1297729 | 1178208 | 424566 | 1,86.38
encoded_numeric_token,68865,10055,396,n720.000 | n900.000 | n56.400 | n3120.000 | n48.000 | n2638.920,13.62
unexpected_token,2,1,2,nInfinity,0.00


,token_type,token,frequencia
6313,encoded_numeric_token,n48.000,3012
896,encoded_numeric_token,n12.000,2567
5030,encoded_numeric_token,n36.000,2342
3273,encoded_numeric_token,n24.000,1868
8447,encoded_numeric_token,n720.000,1518
9811,encoded_numeric_token,n96.000,1518
1851,encoded_numeric_token,n156.000,1466
9182,encoded_numeric_token,n84.000,1002
7228,encoded_numeric_token,n6000.000,934
2424,encoded_numeric_token,n187.200,894


## 7. Perfil semantico por propriedade

A mesma coluna `value` muda de significado conforme `property`. Esta visao mostra o formato dominante de cada propriedade e ajuda a decidir se ela deve virar categoria, token, numero ou flag temporal.

In [63]:
property_grammar_mix = pd.crosstab(value_frame['property'], value_frame['value_grammar'])
property_grammar_pct = property_grammar_mix.div(property_grammar_mix.sum(axis=1), axis=0) * 100

property_value_profile = (
    value_frame.groupby('property')
    .agg(
        linhas=('value', 'size'),
        itens=('itemid', 'nunique'),
        valores_unicos=('value', 'nunique'),
        tokens_media=('token_count', 'mean'),
        pct_multitoken=('is_multitoken', 'mean'),
        pct_encoded_numeric=('encoded_numeric_tokens', lambda values: (values > 0).mean()),
        pct_plain_hash=('plain_hash_tokens', lambda values: (values > 0).mean()),
        exemplos=('value_text', examples_text),
        inicio=('property_time', 'min'),
        fim=('property_time', 'max'),
    )
    .sort_values('linhas', ascending=False)
)

property_value_profile['formato_dominante'] = property_grammar_pct.idxmax(axis=1)
property_value_profile['pct_formato_dominante'] = property_grammar_pct.max(axis=1).round(2)
property_value_profile['tokens_media'] = property_value_profile['tokens_media'].round(3)
for column in ['pct_multitoken', 'pct_encoded_numeric', 'pct_plain_hash']:
    property_value_profile[column] = (property_value_profile[column] * 100).round(2)
property_value_profile['cardinalidade_por_linha'] = (
    property_value_profile['valores_unicos'] / property_value_profile['linhas']
).round(4)

display(
    Markdown(
        'Perfil por propriedade calculado sobre a mesma amostra da classificacao do `value`.'
    )
)
display(property_value_profile.head(35))

Perfil por propriedade calculado sobre a mesma amostra da classificacao do `value`.

,linhas,itens,valores_unicos,tokens_media,pct_multitoken,pct_encoded_numeric,pct_plain_hash,exemplos,inicio,fim,formato_dominante,pct_formato_dominante,cardinalidade_por_linha
property,,,,,,,,,,,,,
888,74047,51792,48941,4.727,73.14,33.39,98.37,298888 1318567 | 428970 65916 6399 n111816.000 373898 784581 1297729 n36.000 350726 30603 832471 | 828472 | 809104 5...,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,multi_hashed_text_tokens,41.25,0.6609
790,44379,32451,7944,1.000,0.00,100.0,0.0,n11280.000 | n88320.000 | n12120.000 | n58980.000 | n204000.000 | n12360.000,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,encoded_numeric_only,100.00,0.1790
available,37137,27749,2,1.000,0.00,0.0,100.0,0 | 1,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,special_available,100.00,0.0001
categoryid,19473,16269,897,1.000,0.00,0.0,100.0,1084 | 498 | 546 | 1674 | 1561 | 421,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,special_categoryid,100.00,0.0461
6,15689,13752,1433,1.883,53.54,0.56,100.0,150169 431733 1037547 | 591638 | 1152934 1238769 | 916238 150169 812456 | 348125 150169 114326 292481 | 719535,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,multi_hashed_text_tokens,52.98,0.0913
283,14702,13171,12813,18.443,100.00,38.03,100.0,741149 1186729 474154 107828 1118082 | 250259 1176791 1135094 1175024 n360.000 | 562911 365855 1305063 | 305351 9372...,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,multi_hashed_text_tokens,61.96,0.8715
776,14078,12637,12573,1.000,0.00,0.0,100.0,409699 | 383750 | 1179847 | 717477 | 922297 | 857377,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,single_hashed_text_token,100.00,0.8931
678,11819,11280,1736,1.208,19.10,0.05,100.0,1318713 | 484516 91207 | 449895 | 1115724 | 708604 | 508354,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,single_hashed_text_token,80.90,0.1469
364,11631,11140,11262,1.000,0.00,0.0,100.0,166696 | 441871 | 766969 | 998058 | 194988 | 184126,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00,single_hashed_text_token,100.00,0.9683


## 8. Propriedades especiais

`categoryid` e `available` sao as duas excecoes importantes: seus valores nao foram hasheados. Elas sao fortes candidatas para a tabela unificada (estudo a parte no 03_retailrocket), desde que o join respeite o timestamp do evento.

In [64]:
special_properties = item_properties[
    item_properties['property'].isin(['categoryid', 'available'])
].copy()

if special_properties.empty:
    display(Markdown('Nenhuma propriedade `categoryid` ou `available` foi encontrada na leitura atual.'))
else:
    special_summary = (
        special_properties.groupby('property')
        .agg(
            linhas=('value', 'size'),
            itens=('itemid', 'nunique'),
            valores_unicos=('value', 'nunique'),
            exemplos=('value', lambda values: ' | '.join(values.dropna().astype(str).drop_duplicates().head(8))),
            inicio=('property_time', 'min'),
            fim=('property_time', 'max'),
        )
        .sort_values('linhas', ascending=False)
    )
    display(special_summary)

    if 'categoryid' in set(special_properties['property']):
        item_categories = special_properties[special_properties['property'].eq('categoryid')].copy()
        item_categories['categoryid'] = pd.to_numeric(item_categories['value'], errors='coerce').astype('Int64')
        category_match = item_categories['categoryid'].isin(category_tree['categoryid'])
        category_validation = pd.DataFrame(
            {
                'metrica': ['linhas_categoryid', 'categorias_unicas', 'pct_presente_na_category_tree'],
                'valor': [
                    len(item_categories),
                    item_categories['categoryid'].nunique(),
                    round(category_match.mean() * 100, 2),
                ],
            }
        )
        display(Markdown('Validacao de `categoryid` contra `category_tree`.'))
        display(category_validation)
        display(item_categories['value'].value_counts().head(20).rename('linhas').to_frame())

    if 'available' in set(special_properties['property']):
        availability = special_properties[special_properties['property'].eq('available')]
        availability_profile = availability['value'].value_counts(dropna=False).rename_axis('available').to_frame('linhas')
        availability_profile['percentual'] = (
            availability_profile['linhas'] / availability_profile['linhas'].sum() * 100
        ).round(2)
        display(Markdown('Distribuicao de disponibilidade documentada: `1` disponivel, `0` indisponivel.'))
        display(availability_profile)

,linhas,itens,valores_unicos,exemplos,inicio,fim
property,,,,,,
available,73869,41905,2,0 | 1,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00
categoryid,38905,27932,976,1338 | 1277 | 1059 | 1147 | 47 | 619 | 1228 | 546,2015-05-10 03:00:00+00:00,2015-09-13 03:00:00+00:00


Validacao de `categoryid` contra `category_tree`.

,metrica,valor
0,linhas_categoryid,38905.00
1,categorias_unicas,976.00
2,pct_presente_na_category_tree,99.98


,linhas
value,
1147,1278
546,1224
491,1040
1613,995
1120,863
342,844
1404,823
1277,714
1167,640


Distribuicao de disponibilidade documentada: `1` disponivel, `0` indisponivel.

,linhas,percentual
available,,
0,42256,57.2
1,31613,42.8


## 9. Resumo interpretativo para modelagem

Esta tabela resume uma primeira leitura de uso potencial. Ela nao decodifica a semantica dos hashes, mas ajuda a separar sinais estruturados, sinais multivalorados e atributos que exigem cautela temporal.

In [65]:
def interpret_property(row: pd.Series) -> str:
    if row.name == 'categoryid':
        return 'excecao documentada; categoria direta do item, ligavel a category_tree'
    if row.name == 'available':
        return 'excecao documentada; flag temporal de disponibilidade 0/1'
    if row['formato_dominante'] == 'mixed_hashed_text_and_numeric':
        return 'texto hasheado com numeros codificados; separar tokens hash e tokens n...'
    if row['formato_dominante'] == 'encoded_numeric_only':
        return 'valor numerico codificado; remover prefixo n e converter para float'
    if row['formato_dominante'] == 'multi_hashed_text_tokens':
        return 'texto multivalorado hasheado; candidato para bag-of-tokens ou embedding'
    if row['formato_dominante'] == 'single_hashed_text_token':
        return 'categoria hasheada de token unico; avaliar cobertura e frequencia'
    return 'formato misto/inesperado; revisar exemplos antes de promover para feature'


def leakage_risk(row: pd.Series) -> str:
    if row.name == 'available' or row['pct_pares_com_mudanca'] > 10:
        return 'alto: exige join temporal por evento'
    if row['cardinalidade_por_linha'] > 0.5:
        return 'medio: alta cardinalidade pode memorizar itens'
    return 'baixo/medio: ainda validar no split cronologico'


candidate_summary = property_value_profile.join(
    stability_by_property[['pct_pares_com_mudanca']], how='left'
).head(30)
candidate_summary['pct_pares_com_mudanca'] = candidate_summary['pct_pares_com_mudanca'].fillna(0)
candidate_summary['leitura_semantica'] = candidate_summary.apply(interpret_property, axis=1)
candidate_summary['risco_modelagem'] = candidate_summary.apply(leakage_risk, axis=1)
candidate_summary['uso_potencial'] = [
    'feature de catalogo ou conteudo' if prop != 'available' else 'feature temporal de disponibilidade'
    for prop in candidate_summary.index
]

display(
    candidate_summary[
        [
            'linhas',
            'itens',
            'valores_unicos',
            'tokens_media',
            'formato_dominante',
            'pct_formato_dominante',
            'pct_multitoken',
            'pct_encoded_numeric',
            'pct_plain_hash',
            'pct_pares_com_mudanca',
            'leitura_semantica',
            'risco_modelagem',
            'uso_potencial',
            'exemplos',
        ]
    ]
)

,linhas,itens,valores_unicos,tokens_media,formato_dominante,pct_formato_dominante,pct_multitoken,pct_encoded_numeric,pct_plain_hash,pct_pares_com_mudanca,leitura_semantica,risco_modelagem,uso_potencial,exemplos
property,,,,,,,,,,,,,,
888,74047,51792,48941,4.727,multi_hashed_text_tokens,41.25,73.14,33.39,98.37,8.19,texto multivalorado hasheado; candidato para bag-of-tokens ou embedding,medio: alta cardinalidade pode memorizar itens,feature de catalogo ou conteudo,298888 1318567 | 428970 65916 6399 n111816.000 373898 784581 1297729 n36.000 350726 30603 832471 | 828472 | 809104 5...
790,44379,32451,7944,1.000,encoded_numeric_only,100.00,0.00,100.0,0.0,16.96,valor numerico codificado; remover prefixo n e converter para float,alto: exige join temporal por evento,feature de catalogo ou conteudo,n11280.000 | n88320.000 | n12120.000 | n58980.000 | n204000.000 | n12360.000
available,37137,27749,2,1.000,special_available,100.00,0.00,0.0,100.0,9.05,excecao documentada; flag temporal de disponibilidade 0/1,alto: exige join temporal por evento,feature temporal de disponibilidade,0 | 1
categoryid,19473,16269,897,1.000,special_categoryid,100.00,0.00,0.0,100.0,5.84,"excecao documentada; categoria direta do item, ligavel a category_tree",baixo/medio: ainda validar no split cronologico,feature de catalogo ou conteudo,1084 | 498 | 546 | 1674 | 1561 | 421
6,15689,13752,1433,1.883,multi_hashed_text_tokens,52.98,53.54,0.56,100.0,2.51,texto multivalorado hasheado; candidato para bag-of-tokens ou embedding,baixo/medio: ainda validar no split cronologico,feature de catalogo ou conteudo,150169 431733 1037547 | 591638 | 1152934 1238769 | 916238 150169 812456 | 348125 150169 114326 292481 | 719535
283,14702,13171,12813,18.443,multi_hashed_text_tokens,61.96,100.00,38.03,100.0,2.91,texto multivalorado hasheado; candidato para bag-of-tokens ou embedding,medio: alta cardinalidade pode memorizar itens,feature de catalogo ou conteudo,741149 1186729 474154 107828 1118082 | 250259 1176791 1135094 1175024 n360.000 | 562911 365855 1305063 | 305351 9372...
776,14078,12637,12573,1.000,single_hashed_text_token,100.00,0.00,0.0,100.0,2.33,categoria hasheada de token unico; avaliar cobertura e frequencia,medio: alta cardinalidade pode memorizar itens,feature de catalogo ou conteudo,409699 | 383750 | 1179847 | 717477 | 922297 | 857377
678,11819,11280,1736,1.208,single_hashed_text_token,80.90,19.10,0.05,100.0,1.00,categoria hasheada de token unico; avaliar cobertura e frequencia,baixo/medio: ainda validar no split cronologico,feature de catalogo ou conteudo,1318713 | 484516 91207 | 449895 | 1115724 | 708604 | 508354
364,11631,11140,11262,1.000,single_hashed_text_token,100.00,0.00,0.0,100.0,1.04,categoria hasheada de token unico; avaliar cobertura e frequencia,medio: alta cardinalidade pode memorizar itens,feature de catalogo ou conteudo,166696 | 441871 | 766969 | 998058 | 194988 | 184126


## 10. Conclusoes para a proxima etapa

- `value` nao e uma coluna unica semanticamente: ela precisa ser lida junto com `property`.
- `categoryid` e `available` sao excecoes nao hasheadas e devem virar features diretas, com join temporal.
- Tokens `n...` representam numeros reais codificados; podem virar features numericas apos remover `n` e converter para `float`.
- Inteiros comuns em propriedades nao especiais representam palavras/valores textuais hasheados; devem ser tratados como tokens categoricos anonimos.
- Valores com varios tokens sao textos/atributos compostos; fazem mais sentido como bag-of-tokens, contagens ou embeddings do que como uma categoria unica.
- Como `item_properties` e um changelog, qualquer atributo deve ser unido aos eventos usando somente valores conhecidos ate o timestamp da interacao.